# SSF2 RL — Implement Your Own Algorithms

A practice space. The environment (`SSF2Env`) and a small offline dataset loader
are provided; **the algorithm bodies are left as TODOs for you to fill in.**

Suggested progression (each is a self-contained section):
1. **Cross-Entropy Method (CEM)** — the simplest policy-search baseline.
2. **REINFORCE** — vanilla policy gradients with a return baseline.
3. **Q-Learning / DQN** — value-based, with a replay buffer and target net.
4. **Behavioral Cloning** — supervised learning from the recorded trajectory
   (`notebooks/data/random_traj.npz`), no environment interaction needed.

Tips:
- Keep episodes short while debugging (`max_episode_frames=30*30`).
- The game runs at 30 FPS in real time, so on-policy learning is slow; prefer
  offline/BC or CEM with small populations first, then scale up.
- Use `torch` (already in the `[rl]` extra) — install with
  `.venv/bin/pip install -e "python[rl]"` if needed.

In [ ]:
import sys
from pathlib import Path

REPO = Path("/Users/cachemiss/Documents/projects/reflash2-fork/reflash2")
if str(REPO / "python") not in sys.path:
    sys.path.insert(0, str(REPO / "python"))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ssf2_rl.env import SSF2Env, ACTION_NAMES

N_ACTIONS = len(ACTION_NAMES)
print("n_actions:", N_ACTIONS)

## Shared utilities

These helpers are given so you can focus on the algorithm math. `rollout` runs
one episode with a policy function `obs -> action` and returns the full
trajectory; `evaluate` averages returns over a few episodes.

In [ ]:
def make_env(**kw):
    return SSF2Env(**kw)


def rollout(env, policy, max_steps=30 * 30):
    """Run one episode. policy: callable obs(np.float32[38]) -> int action.
    Returns dict of lists: obs, actions, rewards, dones."""
    obs, _ = env.reset()
    out = {"obs": [], "actions": [], "rewards": [], "dones": []}
    for _ in range(max_steps):
        a = int(policy(obs))
        obs, r, term, trunc, _ = env.step(a)
        out["obs"].append(obs)
        out["actions"].append(a)
        out["rewards"].append(r)
        done = term or trunc
        out["dones"].append(done)
        if done:
            break
    return out


def evaluate(env, policy, n=3, max_steps=30 * 30):
    """Mean episode return of a policy over n episodes."""
    totals = []
    for _ in range(n):
        tr = rollout(env, policy, max_steps)
        totals.append(sum(tr["rewards"]))
    return float(np.mean(totals))


print("helpers defined")

## Section 1 — Cross-Entropy Method (CEM)

Idea: sample a population of linear-Gaussian policies, keep the top-k by
return, and move the mean toward them.

**TODO (you):**
1. Represent a policy as `theta` of shape `(38, N_ACTIONS)`; action logits = `obs @ theta`.
2. `sample_population(n)` — draw thetas from `N(mean, std)`.
3. `run_cem(iterations, pop_size, elite)` — evaluate each, average elites into the mean.

Start with `pop_size=8, elite=2, iterations=5` and short episodes.

In [ ]:
# --- CEM scaffold: fill in the TODOs ---
OBS_DIM = 38

def cem_policy(theta):
    def policy(obs):
        logits = obs @ theta                      # (N_ACTIONS,)
        return int(np.argmax(logits))
    return policy

def sample_population(mean, std, pop_size):
    # TODO: return a list of pop_size thetas ~ Normal(mean, std)
    raise NotImplementedError

def run_cem(env, iterations=5, pop_size=8, elite=2, std=0.5, max_steps=30*20):
    mean = np.zeros((OBS_DIM, N_ACTIONS))
    history = []
    for it in range(iterations):
        # TODO: 1) sample population
        #       2) evaluate each with rollout() -> sum(rewards)
        #       3) keep top `elite`, set mean = their average
        #       4) record best return
        raise NotImplementedError
    return mean, history

# env = make_env(max_episode_frames=30*20)
# best_theta, hist = run_cem(env)
# print("best returns per iteration:", hist)

## Section 2 — REINFORCE

Train a small MLP stochastic policy with `torch`.

**TODO (you):**
1. `PolicyNet(nn.Module)`: two hidden layers (64), output logits over `N_ACTIONS`.
2. `rollout_torch(env, net)` — sample actions from the categorical, record
   `log_probs` and rewards.
3. Loss = `-sum(log_prob_t * (R_t - b))` where `R_t` are discounted returns
   (gamma ~0.99) and `b` a baseline (mean return works).
4. Track mean return per epoch; plot it.

In [ ]:
# --- REINFORCE scaffold: fill in the TODOs ---
GAMMA = 0.99

class PolicyNet(nn.Module):
    def __init__(self, obs_dim=38, n_actions=N_ACTIONS, hidden=64):
        super().__init__()
        # TODO: define net (e.g. Linear->ReLU->Linear->ReLU->Linear)
        raise NotImplementedError

    def forward(self, x):
        # TODO: return action logits
        raise NotImplementedError

def rollout_torch(env, net, max_steps=30 * 30):
    """Returns (log_probs:list[Tensor], rewards:list[float], done:bool)."""
    obs, _ = env.reset()
    log_probs, rewards = [], []
    for _ in range(max_steps):
        # TODO: forward pass, sample Categorical, record log_prob, env.step
        raise NotImplementedError
    return log_probs, rewards

def discounted_returns(rewards, gamma=GAMMA):
    # TODO: compute R_t = sum_{k>=t} gamma^(k-t) r_k (backwards pass)
    raise NotImplementedError

def train_reinforce(env, epochs=20, max_steps=30 * 30):
    net = PolicyNet()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    returns_history = []
    for ep in range(epochs):
        # TODO: rollout, compute returns, loss = -mean(log_prob * (R - b)),
        #       b = mean of this episode's returns; opt.step()
        raise NotImplementedError
    return net, returns_history

# env = make_env()
# net, hist = train_reinforce(env)

## Section 3 — DQN (value-based)

**TODO (you):**
1. `QNet(nn.Module)`: obs -> Q-values over `N_ACTIONS`.
2. A `ReplayBuffer` (deque of `(s, a, r, s', done)`, `sample(batch)`).
3. Epsilon-greedy collection into the buffer; train with TD targets
   `r + gamma * (1-done) * max_a' Q_target(s')`; soft/hard target updates.

Because the env is real-time (30 FPS), collect in short bursts and reuse the
buffer heavily. Consider `frame-skip` (repeat each action 2-3 frames) to make
learning tractable.

In [ ]:
# --- DQN scaffold: fill in the TODOs ---
class QNet(nn.Module):
    def __init__(self, obs_dim=38, n_actions=N_ACTIONS, hidden=128):
        super().__init__()
        # TODO: define net
        raise NotImplementedError

    def forward(self, x):
        # TODO: return Q-values
        raise NotImplementedError

import collections, random

class ReplayBuffer:
    def __init__(self, capacity=20_000):
        self.buf = collections.deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch):
        # TODO: random.sample and stack into numpy/torch tensors
        raise NotImplementedError

    def __len__(self):
        return len(self.buf)

def train_dqn(env, total_steps=5000, batch=64, gamma=0.99,
              eps_start=1.0, eps_end=0.1, eps_decay=2000):
    # TODO: epsilon-greedy collection + TD training + target updates
    raise NotImplementedError

# env = make_env()
# train_dqn(env)

## Section 4 — Behavioral Cloning (reference, fully runnable)

A working example you can run end-to-end (after recording a trajectory in
notebook 01). It needs no environment interaction, so it's a good first win
and a template for the TODO sections above.

In [ ]:
# --- Behavioral cloning: complete reference ---
TRaj = REPO / "notebooks" / "data" / "random_traj.npz"
assert TRaj.exists(), "run notebook 01 first to record random_traj.npz"

d = np.load(TRaj)
X = torch.tensor(d["obs"], dtype=torch.float32)
y = torch.tensor(d["action"], dtype=torch.long)

bc_net = nn.Sequential(
    nn.Linear(38, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, N_ACTIONS),
)
opt = torch.optim.Adam(bc_net.parameters(), lr=1e-3)

for epoch in range(30):
    opt.zero_grad()
    loss = F.cross_entropy(bc_net(X), y)
    loss.backward()
    opt.step()
    if epoch % 5 == 0:
        acc = (bc_net(X).argmax(1) == y).float().mean().item()
        print(f"epoch {epoch:2d} loss={loss.item():.3f} acc={acc:.2f}")

# Deploy the cloned policy in the live game:
env = make_env()
obs, _ = env.reset()
for _ in range(30 * 10):
    with torch.no_grad():
        a = int(bc_net(torch.tensor(obs)).argmax())
    obs, r, term, trunc, _ = env.step(a)
    if term or trunc:
        obs, _ = env.reset()
env.close()
print("BC policy ran for 10s in the live game.")